## LAB511: Create advanced Postgres-powered agentic apps with Azure HorizonDB

### Part 3.1: Introduction

**Welcome to the LAB511 Agent App Notebook!**

In this notebook, you will build a Agent Framework-based Agent that can reason over our database of legal cases you deployed in the previous steps. You will also incorporate external web service data, and use memory to improve its responses over time.

Agent Framework is an open-source SDK developed by Microsoft that helps developers create advanced AI agents by combining:

- LLMs (Large Language Models) like OpenAI's GPT models
- Agent Function Tools (sometimes called 'AI Function Tools', custom functions the agent can call)
- Memory (saving and recalling past conversations or facts)

An Agent in Agent Framework is a smart assistant that can:

- Respond to user prompts
- Decide which function tools to call
- Use external knowledge sources like databases or APIs
- Build better, grounded answers by combining model reasoning with real-world data

You are about to connect powerful components:

- Azure OpenAI (for embeddings and LLM chat completions)
- Azure PostgreSQL with Vector and Graph extensions (for fast semantic and graph search)
- APIs for real-world data (historical weather evidence)

As you progress, each section of code will incrementally build up these capabilities, and by the final step, you’ll have a highly capable legal research assistant.

**Architecture Diagram**

> The app we are going to build today:

### Part 3.2: Setup the Agent App Python imports

> **Note:** In your lab environment, we already have the PIP packages pre-deployed that are needed by the import statements in the following code block, so you do not need to perform any installs.

##### 🧠 *Technical Background Notes*

This set of imports prepares the technical foundation for building an AI-powered agent that interacts with a PostgreSQL database and Azure OpenAI services. `nest_asyncio` is used to allow nested event loops, which is essential when running asynchronous code inside a Jupyter notebook. Core Python modules like `os`, `asyncio`, `uuid`, and `requests` handle environment access, asynchronous execution, unique ID generation, and external API calls, respectively. `psycopg` provides modern database connectivity to PostgreSQL, while `pydantic` offers structured data validation and modeling through its `Field` component. The `Annotated` type from `typing` enables detailed parameter annotations for function tools. Finally, `dotenv` with `load_dotenv` allows secure loading of environment variables from a `.env` file.

The Agent Framework libraries enable creation of intelligent agents through `AzureOpenAIChatClient` for Azure OpenAI integration, `ChatAgent` for agent orchestration, `ChatMessageStore` for conversation memory, and `ai_function` decorator for registering callable tools. The framework also provides message handling components (`ChatMessage`, `TextContent`, `Role`) for managing conversation state and structure.

##### 📝 *Tasks*

1. Run the cell below using the "▶" icon next to the cell.

1. This will run the code and show the output below.  Since these are just imports, there is nothing to show at the end other than a check mark showing success.

    > **Note:** The first time this code block is ran, it may take around 10-15 seconds.

In [ ]:
# Allow nested event loops (needed so async agent calls work inside Jupyter)
import nest_asyncio
nest_asyncio.apply()

# Standard library
import os
import re
import sys
import io
import json
import time
import asyncio
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

# Type hints used by tool signatures (Annotated drives the JSON schema the agent sees)
from typing import Annotated, Optional, List, Literal

# Third-party: HTTP, data validation, database driver, env loading
import requests
from pydantic import Field
import psycopg
from dotenv import load_dotenv

# Agent Framework: LLM client + the @tool decorator that registers functions as callable tools
from agent_framework.openai import OpenAIChatClient
from agent_framework import tool

# UI + notebook display
import gradio as gr
from IPython.display import Markdown

In [ ]:
load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT   = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]
AZURE_OPENAI_KEY        = os.environ["AZURE_OPENAI_KEY"]
AZURE_EMBED_DEPLOYMENT  = os.environ["AZURE_EMBED_DEPLOYMENT"]
AZURE_API_VERSION       = os.environ["AZURE_API_VERSION"]

DB_CONFIG = {
    # Note we are using the read-only endpoint for the agent as all queries
    # are read-only and is an example of offloading read workloads to a read replica
    "host":     os.environ["AZURE_PG_HOST"],
    "dbname":   os.environ["AZURE_PG_NAME"],
    "user":     os.environ["AZURE_PG_USER"],
    "password": os.environ["AZURE_PG_PASSWORD"],
    "port":     os.environ["AZURE_PG_PORT"],
    "sslmode":  os.environ.get("AZURE_PG_SSLMODE", "require"),
}

print(AZURE_OPENAI_DEPLOYMENT)
print(AZURE_EMBED_DEPLOYMENT)
print(AZURE_API_VERSION)

## Tool 1

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Tool 1: keyword_case_search
# ─────────────────────────────────────────────────────────────────────
# This is the FIRST tool in our agent's legal research pipeline.
#
# Purpose:
#   Perform fast, lexical (keyword-based) full-text search over Washington
#   State case law opinions using the `pg_fts` PostgreSQL extension, which
#   implements BM25 ranking (the same algorithm family used by Lucene /
#   Elasticsearch / Tantivy). This is excellent for surfacing canonical
#   precedents when the user (or agent) already knows the legal doctrine
#   name, a citation, or distinctive terminology (e.g., "common enemy
#   doctrine").
#
# Why BM25 instead of (or in addition to) vector search?
#   BM25 excels at exact-term and phrase matching. Vector search (Tool 2)
#   excels at semantic / conceptual similarity. Using BOTH gives the agent
#   broader recall — keyword anchors the canonical cases, semantic
#   expands to factually similar ones.
#
# The @tool decorator (from agent_framework) registers this Python
# function as a callable tool for the LLM agent. The decorator inspects
# the function signature plus Annotated[..., Field(description=...)]
# metadata to build the JSON schema the agent sees when deciding whether
# to call this tool and what arguments to pass.
# ─────────────────────────────────────────────────────────────────────
@tool(description=(
    "BM25 full-text search over Washington case law opinions. "
    "Supports phrase proximity ('common enemy'~3), boolean operators "
    "(AND, OR, NOT), and optional filters by court level and minimum decision date. "
    "Returns the top matching cases ranked by BM25 relevance score. "
    "Use this first to surface canonical precedents by doctrine name or citation."
))
def keyword_case_search(
    # The main search query. Tantivy/BM25 syntax supports:
    #   - Plain terms:  surface water drainage
    #   - Phrases:      "surface water"
    #   - Proximity:    "common enemy"~3   (within 3 tokens of each other)
    #   - Booleans:     AND, OR, NOT
    #   - Grouping:     (regrade OR grading)
    query: Annotated[str, Field(
        description=(
            "BM25/Tantivy-syntax query over case opinions. "
            "Examples: 'common enemy doctrine', "
            "'\"surface water\"~3 AND (regrade OR grading)', "
            "'qualified immunity NOT prison'."
        )
    )],
    # Optional metadata filter — restrict to one court tier. Useful when
    # the agent wants only binding precedent (Supreme Court) vs.
    # persuasive authority (Court of Appeals).
    court_level: Annotated[Optional[str], Field(
        default=None,
        description=(
            "Optional filter: 'Washington Supreme Court' "
            "or 'Washington Court of Appeals'. "
            "None = both courts."
        ),
    )] = None,
    # Optional metadata filter — exclude cases older than `min_year`.
    # Helpful for narrowing to modern doctrine while still allowing access
    # to historical canonical cases when min_year is None or very low.
    min_year: Annotated[Optional[int], Field(
        default=None,
        description="Optional filter: only return cases decided on or after this year."
    )] = None,
) -> str:
    # Diagnostic logging — printed to stdout so the Gradio UI's
    # "Tool Trace" panel can detect that this tool was invoked and display
    # its parameters. The trace parser keys off the literal string
    # "<tool_name> was called", so keep that line shape stable.
    print("keyword_case_search was called")
    print(f"  query={query!r}  court_level={court_level}  min_year={min_year}  limit=5")

    # Open a short-lived connection to Azure Database for PostgreSQL.
    # autocommit=True avoids needing an explicit COMMIT for SET statements.
    # Using a context manager ensures the connection + cursor are closed
    # cleanly even if an exception is raised mid-query.
    with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
        # Initialize pg_fts for this session (temporary step during preview).
        cur.execute("SELECT pgfts.hello_pg_fts();")

        # Add the pgfts schema to the session search_path so we can call
        # fts_query() and fts_score() unqualified in the SELECT below.
        cur.execute("SET search_path = public, pgfts;")

        # ──────────────────────────────────────────────────────────────
        # Build the WHERE clause dynamically so optional filters
        # (court_level, min_year) only appear when provided. The first
        # predicate is the BM25 match against the pre-built FTS index
        # named 'idx_cases_fts' (created in notebook 1). Parameters are
        # passed via psycopg's parameter binding (%s) — never via string
        # concatenation — to keep the query safe from SQL injection.
        # ──────────────────────────────────────────────────────────────
        where = ["fts_query(%s, 'idx_cases_fts')"]
        params = [query]
        if court_level:
            where.append("court_level = %s")
            params.append(court_level)
        if min_year:
            # decision_date is a DATE column; compare against Jan 1 of the
            # requested year to act as a "year >= min_year" filter.
            where.append("decision_date >= %s")
            params.append(f"{min_year}-01-01")

        # Execute the search. fts_score(opinion) returns the BM25
        # relevance score for the matched row, which we surface to the
        # agent so it can reason about how strong each match is.
        # LIMIT 5 keeps the response small enough for the LLM context
        # window and bounds latency.
        cur.execute(
            f"""
            SELECT id, name, decision_date, court_level,
                   fts_score(opinion) AS score, opinion
            FROM cases
            WHERE {' AND '.join(where)}
            LIMIT 5;
            """,
            tuple(params),
        )
        rows = cur.fetchall()

    # Early return when the query matched nothing — keeps downstream
    # parsing (the regex that pulls case ids out of each line) simple.
    if not rows:
        return "No matches"

    # ──────────────────────────────────────────────────────────────────
    # Format results as a pipe-delimited text block. This shape matters:
    # downstream code (the parse_ids helper used by later test cells and
    # by Tool 3's seed extraction) uses a regex like  ^(\d+)\s+\|  to
    # extract case ids from the first column. If you change the format,
    # update those parsers too.
    #
    # The opinion is truncated to 300 chars to keep the agent's context
    # window manageable while still giving it enough text to reason over.
    # Newlines in the opinion are flattened so each row stays on one
    # line — again, to keep the line-based parser working.
    # ──────────────────────────────────────────────────────────────────
    lines = []
    for case_id, name, decision_date, level, score, opinion in rows:
        snippet = (opinion or "").replace("\n", " ")[:300]
        lines.append(
            f"{case_id} | {decision_date} | {level} | score={score:.3f} | "
            f"{name}: {snippet}..."
        )
    return "\n".join(lines)


In [ ]:
print("// Test Call of keyword_case_search //")
print(keyword_case_search("surface water OR flooding OR drainage"))

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# First Agent Assembly: wire the LLM up to Tool 1 and run a query
# ─────────────────────────────────────────────────────────────────────
# So far we've just defined and unit-tested keyword_case_search as a
# plain Python function. In this cell we promote it into an actual
# agent: an LLM that can autonomously decide WHEN to call the tool,
# WHAT arguments to pass, and HOW to weave the tool's output into a
# natural-language answer for the user.
#
# This is the "smallest interesting" version of the agent — one tool,
# one prompt. Later cells will add semantic search (Tool 2), the
# citation graph (Tool 3), entity extraction (Tool 4), and weather
# evidence (Tool 5), then reassemble the agent each time so you can
# see how additional tools change its behavior.
# ─────────────────────────────────────────────────────────────────────

# Build the underlying chat client.
#
# Despite the class name `OpenAIChatClient`, this is configured to talk
# to Azure OpenAI by passing `azure_endpoint`. The Agent Framework SDK
# routes requests to your Azure OpenAI deployment using:
#   - model:           the Azure OpenAI deployment name (NOT the base
#                      model name like "gpt-4o"). This is the name you
#                      gave the deployment in Azure AI Foundry / the
#                      Azure portal.
#   - azure_endpoint:  the resource endpoint, e.g.
#                      https://<your-resource>.openai.azure.com/
#   - api_key:         the resource key. In production prefer Azure AD /
#                      Managed Identity over a static key.
client = OpenAIChatClient(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_KEY"],
)

# Turn the chat client into an Agent.
#
# `as_agent(...)` wraps the chat client in an agent loop that:
#   1. Sends the user message + the system `instructions` to the LLM.
#   2. Advertises the `tools` list to the LLM as callable functions
#      (the JSON schema is auto-generated from each @tool-decorated
#      function's signature and Annotated[..., Field(description=...)]
#      metadata — that's why those descriptions matter).
#   3. If the LLM responds with a tool call, the framework executes the
#      Python function, feeds the result back into the conversation,
#      and lets the model continue reasoning.
#   4. Repeats until the model produces a final natural-language answer.
#
# `instructions` is the system prompt. Keep it short and directive:
#   - Tells the model its role ("legal assistant").
#   - Tells it the desired output shape (case names + why relevant).
#   - "Run each tool only once" discourages the model from re-calling
#     the same tool with slightly tweaked arguments, which wastes
#     latency and token budget without improving recall.
agent = client.as_agent(
    instructions=(
        "You are a helpful legal assistant. Respond with the case names, and why each is relevant. Run each tool only once."
    ),
    tools=[keyword_case_search],
)

# The user question. Notice we do NOT pre-extract keywords or hand the
# tool a search string directly — that's the agent's job. The LLM will
# read this natural-language prompt, decide which terms to search for
# (e.g., "surface water", "drainage", "common enemy doctrine"), and
# call keyword_case_search with arguments of its own choosing.
user_query = (
    "I have a client in Seattle, WA whose property keeps flooding after the developer "
    "next door regraded their lot and the city redid the drainage on the "
    "road."
)

# Run the agent. `agent.run(...)` is async, so we `await` it. Because
# we called `nest_asyncio.apply()` at the top of the notebook, awaiting
# at the top level of a cell works inside Jupyter.
#
# While this runs, watch the cell output: the `print(...)` lines inside
# keyword_case_search (e.g., "keyword_case_search was called") will
# appear, which is how you confirm the agent actually invoked the tool
# rather than hallucinating an answer from prior knowledge.
print("// Functions the Agent Called: //")
result = await agent.run(user_query)

# `result.text` is the agent's final natural-language answer (after any
# tool calls have been folded in). We render it as Markdown so case
# names, bullets, and formatting display nicely in the notebook.
print("")
print("// Agent Response: //")
Markdown(result.text)


## Tool 2

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Tool 2: semantic_case_search
# ─────────────────────────────────────────────────────────────────────
# This is the SECOND tool in our agent's legal research pipeline.
#
# Purpose:
#   Find cases that are *conceptually* similar to a natural-language
#   fact pattern, even when the user doesn't know the legal doctrine
#   name or canonical citation. Tool 1 (keyword_case_search) is great
#   when you already know the right terms ("common enemy doctrine");
#   Tool 2 shines when you only know the situation ("my client's lot
#   floods after a neighbor regraded their land").
#
# Pipeline role / how results combine with Tool 1:
#   - Tool 1 returns 5 cases by BM25 keyword relevance.
#   - Tool 2 returns 5 cases by vector similarity (INDEPENDENTLY of
#     Tool 1 — this function does NOT look at or merge Tool 1's
#     results).
#   - The CALLER (the agent, or the test cells below) takes the UNION
#     of those id sets (up to 10 unique cases) and passes them to
#     Tool 3 as `seed_case_ids` for graph expansion.
#
# DiskANN Advanced Filtering — always on:
#   This lab is specifically about showcasing DiskANN's advanced
#   filtering, which pushes WHERE predicates INTO the index traversal
#   rather than applying them as a post-filter. To make sure the
#   feature is exercised on every call, the session GUCs are set
#   unconditionally below — there is no opt-out flag.
#
# Required filters (court_level, min_year):
#   These are deliberately REQUIRED parameters (not Optional) so the
#   LLM always supplies values. That, in turn, guarantees the WHERE
#   clause has predicates for DiskANN advanced filtering to act on,
#   so the lab consistently demonstrates filtered vector search rather
#   than degenerating into an unfiltered top-k.
#
#   - court_level accepts "Both" to mean "do not filter by court"; the
#     model has no ambiguous "leave it out" option.
#   - min_year accepts a low year like 1900 to mean "no cutoff".
# ─────────────────────────────────────────────────────────────────────

# Whitelist of acceptable court_level filter values. "Both" is the
# explicit "no court filter" value — we validate against this set so a
# typo from the LLM produces a clean error message instead of silently
# returning zero rows.
VALID_COURT_LEVELS = {"Washington Supreme Court", "Washington Court of Appeals", "Both"}

@tool(description=(
    "Semantic vector search over Washington case opinions using DiskANN with "
    "Advanced Filtering enabled (filters are pushed into the index traversal). "
    "Returns the top 5 cases ranked by cosine similarity to the query embedding. "
    "Both court_level and min_year are REQUIRED — pass court_level='Both' to search "
    "across both courts, and pass min_year=1900 if you do not want a date cutoff. "
    "This tool runs independently of keyword_case_search; the caller is expected to "
    "take the UNION of ids from both tools before passing them to precedent_graph_search."
))
def semantic_case_search(
    # The natural-language query. Unlike Tool 1, no special syntax —
    # the embedding model captures meaning, not literal tokens. So
    # "developer regraded lot causing flooding" works as well as a
    # carefully crafted boolean expression would in BM25.
    query_text: Annotated[str, Field(
        description="Natural language query describing the fact pattern or legal issue."
    )],
    # REQUIRED court filter. The model must pick one of the three
    # values. "Both" is the explicit way to skip filtering by court so
    # the model never has to omit the argument.
    court_level: Annotated[Literal["Washington Supreme Court", "Washington Court of Appeals", "Both"], Field(
        description=(
            "REQUIRED court filter. Use 'Washington Supreme Court' for binding precedent only, "
            "'Washington Court of Appeals' for that court only, or 'Both' to search across both courts."
        ),
    )],
    # REQUIRED year-floor filter. The model must pass an integer; use
    # 1900 (or earlier) to effectively disable the cutoff while still
    # giving DiskANN's advanced filtering a predicate to work with.
    min_year: Annotated[int, Field(
        description=(
            "REQUIRED minimum decision year. Only cases decided on or after Jan 1 of this year "
            "are returned. Pass 1900 to effectively disable the cutoff."
        )
    )],
) -> str:
    # Diagnostic trace line — the Gradio UI's Tool Trace panel keys off
    # the literal "<tool_name> was called" string, so keep that shape.
    print("semantic_case_search was called")
    print(
        f"  query_text={query_text!r}  court_level={court_level}  "
        f"min_year={min_year}  limit=5  advanced_filtering=ALWAYS_ON"
    )

    # Validate against the whitelist. Returning early with a clear
    # message helps the LLM self-correct on its next turn.
    if court_level not in VALID_COURT_LEVELS:
        return f"Invalid court_level {court_level!r}. Use one of: {sorted(VALID_COURT_LEVELS)}"

    # Short-lived autocommit connection — SET statements affect only
    # this session, so a fresh connection per call gives us a clean
    # GUC state every time.
    with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
        # ──────────────────────────────────────────────────────────────
        # DiskANN advanced-filtering GUCs (session-level) — ALWAYS ON.
        # These tell the DiskANN index to evaluate the WHERE predicates
        # DURING graph traversal rather than as a post-filter. The
        # values below are sensible defaults for this lab; tune per
        # workload in production.
        #
        #   enable_filter_hook    – master switch for advanced filtering.
        #   selectivity_min/_threshold – when to engage filtered traversal
        #         based on estimated predicate selectivity (fraction of
        #         rows expected to match). 0.0..1.0 means "always try".
        #   filtering_beta        – tradeoff between recall and latency
        #         when navigating the filtered subgraph. Higher = more
        #         exploration = better recall, slightly slower.
        #   l_value_is            – search list size for the in-memory
        #         portion of the index. Larger = better recall, slower.
        # ──────────────────────────────────────────────────────────────
        cur.execute("SET diskann.enable_filter_hook = 'true';")
        cur.execute("SET diskann.selectivity_min = '0.0';")
        cur.execute("SET diskann.selectivity_threshold = '1.0';")
        cur.execute("SET diskann.filtering_beta = 0.85;")
        cur.execute("SET diskann.l_value_is = 300;")

        # ──────────────────────────────────────────────────────────────
        # Build the WHERE clause. We use NAMED placeholders (%(name)s)
        # so we can pass a single params dict.
        #
        # `opinions_vector IS NOT NULL` excludes any case where the
        # embedding wasn't generated (defensive — shouldn't happen in
        # this lab, but cheap insurance).
        #
        # court_level filter is added unless the model passed "Both"
        # (the explicit "no court filter" sentinel). min_year is always
        # applied — pass 1900 for "no cutoff".
        # ──────────────────────────────────────────────────────────────
        where = [
            "opinions_vector IS NOT NULL",
            "decision_date >= make_date(%(min_year)s, 1, 1)",
        ]
        params = {
            "q": query_text,
            "lim": 5,
            "min_year": min_year,
            # The embedding deployment name (e.g. "text-embedding-3-small")
            # is what azure_openai.create_embeddings() needs to know which
            # Azure OpenAI deployment to call. The endpoint + key were
            # configured back in Notebook 1 via azure_ai.set_setting().
            "embed_deployment": AZURE_EMBED_DEPLOYMENT,
        }

        if court_level != "Both":
            where.append("court_level = %(court_level)s")
            params["court_level"] = court_level

        # ------------------------------------------------------------------
        # Embedding call: pick ONE of the two approaches below.
        #
        # (A) AI Model Management (managed alias) -- COMMENTED OUT for now.
        #     Uses azure_ai.create_embeddings() with a configured model alias
        #     (e.g. 'default-embedding') that maps to a backing deployment via
        #     AI Model Management or BYOM model_registry.model_add. Leave this
        #     here so we can switch back when managed models are enabled.
        #
        #     embed_sql = (
        #         "azure_ai.create_embeddings('default-embedding', %(q)s)::vector"
        #     )
        #
        # (B) Traditional azure_ai extension (ACTIVE).
        #     Uses azure_openai.create_embeddings(<deployment>, <text>) which
        #     reads the endpoint + subscription key set in Notebook 1 via
        #     azure_ai.set_setting('azure_openai.endpoint'/'subscription_key').
        #     The deployment name comes straight from AZURE_EMBED_DEPLOYMENT.
        # ------------------------------------------------------------------
        # `::vector` casts the JSONB array returned by the function into
        # pgvector's `vector` type so the `<=>` operator below works.
        embed_sql = (
            "azure_openai.create_embeddings(%(embed_deployment)s, %(q)s)::vector"
        )

        # ──────────────────────────────────────────────────────────────
        # The core semantic-search query.
        #
        #   `opinions_vector <=> <query_vector>`  – cosine DISTANCE
        #       (NOT similarity). Smaller = more similar. 0.0 is a
        #       perfect match; 2.0 is opposite. Other pgvector operators:
        #         <->  L2 distance
        #         <#>  negative inner product
        #
        #   ORDER BY distance ASC + LIMIT 5  – this is the pattern the
        #   DiskANN planner recognizes to engage the index. Without an
        #   ORDER BY on the distance expression it would fall back to a
        #   sequential scan, which is dramatically slower on large
        #   corpora.
        # ──────────────────────────────────────────────────────────────
        cur.execute(
            f"""
            SELECT id, name, decision_date, court_level,
                   opinions_vector <=> {embed_sql} AS distance,
                   opinion
            FROM public.cases
            WHERE {' AND '.join(where)}
            ORDER BY distance ASC
            LIMIT %(lim)s;
            """,
            params,
        )
        rows = cur.fetchall()

    # Nothing matched.
    if not rows:
        return "No matches"

    # ──────────────────────────────────────────────────────────────────
    # Format the response. Same pipe-delimited shape as Tool 1 so the
    # `parse_ids` helper used in later cells can extract case ids from
    # either tool's output with the same regex (^(\d+)\s+\|).
    # Snippets are flattened to one line and truncated to 300 chars to
    # control LLM context size.
    #
    # NOTE: This output contains only the 5 vector-search hits. To feed
    # Tool 3, take the UNION of these ids with Tool 1's ids — see the
    # test cell below for the pattern.
    # ──────────────────────────────────────────────────────────────────
    lines = []
    for case_id, name, decision_date, level, dist, opinion in rows:
        snippet = (opinion or "").replace("\n", " ")[:300]
        dist_str = "n/a" if dist is None else f"{dist:.4f}"
        lines.append(
            f"{case_id} | {decision_date} | {level} | distance={dist_str} | "
            f"{name}: {snippet}..."
        )

    return "\n".join(lines)


## Test Tool 1 and 2 together directly

In [ ]:
print("// Step 1: Keyword anchors (BM25) - returns up to 5 cases //")
kw = keyword_case_search(
    "'common enemy'~3 AND (surface water OR drainage OR stormwater)",
    court_level="Washington Supreme Court",
    min_year=1900,
)
print(kw)

# Pull ids from the formatted output (each line starts with "<id> | ...")
kw_ids = []
for line in kw.splitlines():
    m = re.match(r"^(\d+)\s+\|", line)
    if m:
        kw_ids.append(int(m.group(1)))

print("")
print("// Step 2: Semantic expansion (DiskANN, Advanced Filtering on) - returns up to 5 cases //")
flagship_prompt = (
    "I have a client whose property keeps flooding after the developer next door "
    "regraded their lot and the city redid the drainage on the road."
)
sem = semantic_case_search(
    flagship_prompt,
    court_level="Both",   # "Both" = search across both courts (required arg, no None)
    min_year=1900,        # 1900 = effectively no cutoff (required arg)
)
print(sem)

# Same regex against Tool 2's output to extract its 5 ids.
sem_ids = []
for line in sem.splitlines():
    m = re.match(r"^(\d+)\s+\|", line)
    if m:
        sem_ids.append(int(m.group(1)))

print("")
print("// Step 3 preview: union of Tool 1 + Tool 2 ids -> seeds for Tool 3 //")
# Tool 3 will consume the UNION of ids from Tool 1 + Tool 2 (up to 10 unique cases).
seed_ids = sorted(set(kw_ids) | set(sem_ids))
print(f"Tool 1 ids ({len(kw_ids)}): {kw_ids}")
print(f"Tool 2 ids ({len(sem_ids)}): {sem_ids}")
print(f"Union  ids ({len(seed_ids)}): {seed_ids}")


## Test Tool 1 and 2 together by re-assembling our Agent and running the same prompt

In [ ]:
client = OpenAIChatClient(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_KEY"],
)

agent = client.as_agent(
    instructions=(
        "You are a helpful legal assistant. Respond with the case names, why each is relevant. Run each tool only once."
    ),
    tools=[keyword_case_search,semantic_case_search]
)

user_query = (
    "I have a client in Seattle, WA whose property keeps flooding after the developer "
    "next door regraded their lot and the city redid the drainage on the "
    "road."
)

print("// Functions the Agent Called: //")
result = await agent.run(user_query)

print("")
print("// Agent Response: //")
Markdown(result.text)

## Tool 3 - Graph

In [ ]:
@tool(description=(
    "Traverse the Washington case citation graph stored in Apache AGE (graph name: case_graph). "
    "Edges are (a)-[:REF]->(b) meaning 'a cites b'. "
    "Given seed case ids (from keyword_case_search and semantic_case_search), "
    "returns the TOP 10 most relevant related precedent cases (by centrality score) via a "
    "2-hop bidirectional citation traversal (follows both 'cites' and 'cited-by' edges), "
    "enriched with metadata from the relational cases table. "
    "Washington Supreme Court cases are always boosted in the ranking as binding precedent."
))
def precedent_graph_search(
    seed_case_ids: Annotated[List[int], Field(
        description=(
            "List of seed case ids from Tool 1 and/or Tool 2. "
            "Pass the union of ids from both tools for best coverage."
        )
    )],
) -> str:
    # Cap the number of related cases returned, so Tool 4 (extract) doesn't
    # get flooded with low-relevance hits. We rank by `score` (see below)
    # and take the top N.
    TOP_K = 10

    # Court-weight boost for Washington Supreme Court (binding precedent).
    # Always applied — there is no opt-out, because binding precedent
    # should always outrank persuasive Court of Appeals authority when
    # path counts are otherwise comparable.
    SUPREME_COURT_BOOST = 2

    # Fixed traversal parameters — kept inside the function so the LLM
    # doesn't have to think about them and the lab demo stays consistent.
    #
    #   HOPS = 2  → up to cite-of-cite expansion. 1 would be direct
    #               citations only (often too narrow). 3 starts pulling
    #               in weakly-related cases and dramatically increases
    #               result size.
    #   DIRECTION = "both" → follow both outgoing edges (seed cites prior
    #               authority) AND incoming edges (later cases cite the
    #               seed). The union gives the most complete picture of
    #               the line of authority around the seed cluster.
    HOPS = 2

    print("precedent_graph_search was called")
    print(f"  seeds={len(seed_case_ids)}  direction=both  hops={HOPS}  top_k={TOP_K}  supreme_boost={SUPREME_COURT_BOOST}")

    if not seed_case_ids:
        return "No seed_case_ids provided."

    # Graph stores case_id as text property, created from temp_cases.data->>'id'
    # Your helper functions store it as text, so we compare using string literals.
    seed_literals = ", ".join([f"'{int(i)}'" for i in sorted(set(seed_case_ids))])

    # Build Cypher fragments for both directions of the REF edge.
    # REF direction: (a)-[:REF]->(b) means "a cites b"
    #   - "cites":    seed -> authority           (outgoing from seed)
    #   - "cited_by": later_case -> seed          (incoming to seed,
    #                                              traversed by matching
    #                                              (t)-[:REF*]->(s))
    # We always run both and aggregate path counts together.
    cypher_queries = [
        f"""
            SELECT * FROM cypher('case_graph', $$
                MATCH (s:case)-[:REF*1..{HOPS}]->(t:case)
                WHERE s.case_id IN [{seed_literals}]
                RETURN t.case_id AS related_id, count(*) AS paths
            $$) AS (related_id agtype, paths agtype);
        """,
        f"""
            SELECT * FROM cypher('case_graph', $$
                MATCH (t:case)-[:REF*1..{HOPS}]->(s:case)
                WHERE s.case_id IN [{seed_literals}]
                RETURN t.case_id AS related_id, count(*) AS paths
            $$) AS (related_id agtype, paths agtype);
        """,
    ]

    # Execute graph traversal and aggregate path counts.
    related_paths = {}  # related_id(int) -> paths(int)
    try:
        with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
            # AGE session setup            
            cur.execute("SET search_path = public, ag_catalog, \"$user\";")

            for q in cypher_queries:
                cur.execute(q)
                for related_id_ag, paths_ag in cur.fetchall():
                    # AGE returns agtype; psycopg gives it as Python str like '"123"' or '123'
                    rid_str = str(related_id_ag).strip('"')
                    if not rid_str.isdigit():
                        continue
                    rid = int(rid_str)

                    # Skip seeds in the related set; we only want expansions
                    if rid in seed_case_ids:
                        continue

                    p = int(str(paths_ag).strip('"')) if str(paths_ag).strip('"').isdigit() else 1
                    related_paths[rid] = related_paths.get(rid, 0) + p

    except Exception as e:
        return f"Graph query failed. Is case_graph created and populated? Error: {e}"

    if not related_paths:
        return "No related cases found from the citation graph for the provided seeds."

    related_ids = list(related_paths.keys())

    # Enrich from relational table so Tool 4 can immediately extract from opinion text.
    # Also compute a simple ranking score: paths (centrality proxy) + court weighting.
    try:
        with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
            cur.execute(
                """
                SELECT id, name, decision_date, court_level, opinion
                FROM public.cases
                WHERE id = ANY(%s);
                """,
                (related_ids,)
            )
            rows = cur.fetchall()
    except Exception as e:
        return f"Failed to fetch case metadata from relational table. Error: {e}"

    # Court weighting: WA Supreme Court always gets the binding-precedent
    # boost; Court of Appeals (and anything else) gets 0.
    def court_weight(level: str) -> int:
        if level == "Washington Supreme Court":
            return SUPREME_COURT_BOOST
        return 0

    # ──────────────────────────────────────────────────────────────────
    # Score = paths + court_weight
    #
    #   paths        = count of distinct multi-hop citation paths from
    #                  any seed to this case (a centrality proxy — more
    #                  paths = the case sits closer to the seed cluster
    #                  in the citation graph).
    #   court_weight = +2 if Washington Supreme Court, else 0. Coarse
    #                  boost for binding precedent over Court of Appeals.
    #                  Always applied (no opt-out).
    #
    # Ties on `score` are broken by raw `paths`.
    # ──────────────────────────────────────────────────────────────────
    ranked = []
    for case_id, name, decision_date, court_level, opinion in rows:
        paths = related_paths.get(case_id, 0)
        score = paths + court_weight(court_level)
        snippet = (opinion or "").replace("\n", " ")[:240]
        ranked.append((score, paths, case_id, decision_date, court_level, name, snippet))

    ranked.sort(key=lambda x: (x[0], x[1]), reverse=True)

    # Keep only the top-K highest-scoring related cases. This bounds the
    # payload size handed off to Tool 4 (extract) and to the LLM context
    # window, while still keeping the most central / binding precedents.
    total_related = len(ranked)
    ranked = ranked[:TOP_K]

    print(f"  related_count_total={total_related}  returned_top_k={len(ranked)}")

    # Return a parseable, stable text format:
    # - first section: ids only (for Tool 4 handoff) — already trimmed to top-K
    # - second section: human-readable justification
    top_ids = [str(r[2]) for r in ranked]
    lines = []
    lines.append("RELATED_CASE_IDS: " + ", ".join(top_ids))
    lines.append("")
    lines.append(f"Related cases (top {len(ranked)} of {total_related}, ranked by score):")
    for score, paths, case_id, decision_date, court_level, name, snippet in ranked:
        lines.append(
            f"{case_id} | {decision_date} | {court_level} | paths={paths} | score={score} | {name}: {snippet}..."
        )

    return "\n".join(lines)


## Test Tool 1, 2 and 3 together Directly

In [ ]:
def parse_ids(tool_output: str) -> list[int]:
    ids = []
    for line in tool_output.splitlines():
        m = re.match(r"^(\d+)\s+\|", line)
        if m:
            ids.append(int(m.group(1)))
    return ids

# Example: run Tool 1 and Tool 2 quickly (or reuse saved strings)
kw_out = keyword_case_search(
    "'common enemy'~3 AND (surface water OR drainage OR stormwater)",
    court_level="Washington Supreme Court",
    min_year=1900,
)

sem_out = semantic_case_search(
    "developer regraded lot causing flooding; city changed road drainage",
    court_level="Both",   # "Both" = search across both courts (required arg)
    min_year=1900,        # 1900 = effectively no cutoff (required arg)
)

seed_ids = sorted(set(parse_ids(kw_out) + parse_ids(sem_out)))
print("Seed ids:", seed_ids)

print("")
print("// Tool 3: Graph traversal //")
graph_out = precedent_graph_search(seed_case_ids=seed_ids)
print(graph_out)


## Re-assmble Agent and integrate Tool 3

In [ ]:
client = OpenAIChatClient(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_KEY"],
)

agent = client.as_agent(
    instructions=(
        "You are a helpful legal assistant. Respond with the case names, why each is relevant. Run each tool only once."
    ),
    tools=[keyword_case_search,semantic_case_search,precedent_graph_search]
)

user_query = (
    "I have a client in Seattle, WA whose property keeps flooding after the developer "
    "next door regraded their lot and the city redid the drainage on the road."
)

print("// Functions the Agent Called: //")
result = await agent.run(user_query)

print("")
print("// Agent Response: //")
Markdown(result.text)

## Tool 4 - Extract() entitied from results of found cases

In [ ]:
@tool(description=(
    "Extract structured legal facts from one or more case opinions using azure_ai.extract. "
    "Returns JSON per case with compact fields (holding and issues by default). "
    "Use after the graph tool to turn raw opinions into brief-ready structured records."
))
def case_analyst_extract(
    case_ids: Annotated[List[int], Field(
        description=(
            "List of case ids to analyze (typically from precedent_graph_search's RELATED_CASE_IDS)."
        )
    )]    
) -> str:
    
    # Set chat deployment
    chat_deployment = AZURE_OPENAI_DEPLOYMENT

    print("case_analyst_extract was called")
    print(f"  case_ids={len(case_ids)}  chat_deployment={chat_deployment}")

    if not case_ids:
        return "CASE_BRIEFS_JSON: []"

    # Fields to Extract (always use the default set)
    fields = ["holding", "issues", "statutes_cited", "disposition"]

    # Emit the entity fields the agent is extracting (parsed by the UI trace panel)
    print(f"  fields={','.join(fields)}")

    max_opinion_chars = 1200

    # Step 1: Fetch case rows in a single query
    with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
        cur.execute("CREATE EXTENSION IF NOT EXISTS azure_ai;")
        cur.execute(
            """
            SELECT id, name, decision_date, court_level, opinion
            FROM public.cases
            WHERE id = ANY(%s);
            """,
            (case_ids,),
        )
        rows = cur.fetchall()

    if not rows:
        return "CASE_BRIEFS_JSON: []"

    # Step 2: Extract in parallel — each thread opens its own connection
    # so the LLM calls run concurrently instead of sequentially.
    #
    # Extraction call: pick ONE of the two approaches below.
    #
    # (A) AI Model Management (managed alias) -- COMMENTED OUT for now.
    #     Uses azure_ai.extract(text, labels, model_alias) where the third
    #     argument is a configured alias (e.g. 'default-chat') that maps to
    #     a chat deployment via AI Model Management. Leave this here so we
    #     can switch back when managed models are enabled.
    #
    
    extract_sql = "SELECT azure_ai.extract(%s::text, %s::text[], %s::text);"
    extract_model = "default-chat"
    
    #
    # (B) Traditional azure_ai extension (ACTIVE).
    #     Uses azure_ai.extract(text, labels, <deployment>) which reads
    #     the endpoint + subscription key set in Notebook 1 via
    #     azure_ai.set_setting('azure_openai.endpoint'/'subscription_key').
    #     The deployment name comes from AZURE_OPENAI_DEPLOYMENT.
    #     Explicit ::text / ::text[] casts are required because psycopg sends
    #     parameters as 'unknown' and PG can't pick an overload otherwise.
    
    #extract_sql = "SELECT azure_ai.extract(%s::text, %s::text[], %s::text);"
    #extract_model = chat_deployment

    def _extract_one(row):
        case_id, name, decision_date, court_level, opinion = row
        short_opinion = (opinion or "")[:max_opinion_chars]
        try:
            with psycopg.connect(**DB_CONFIG, autocommit=True) as c2, c2.cursor() as cur2:
                cur2.execute(
                    extract_sql,
                    (short_opinion, fields, extract_model),
                )
                extracted = cur2.fetchone()[0]
        except Exception as e:
            extracted = {"error": "extraction_failed", "reason": str(e)}

        card = {
            "id": case_id,
            "name": name,
            "decision_date": str(decision_date),
            "court_level": court_level,
            "extracted": extracted,
        }
        return card

    with ThreadPoolExecutor(max_workers=len(rows)) as pool:
        futures = {pool.submit(_extract_one, r): r[0] for r in rows}
        results = []
        for fut in as_completed(futures):
            card = fut.result()
            results.append(card)
            # Emit per-case completion line for the UI trace panel
            cname = (card.get("name") or "").replace("|", "/")[:60]
            print(f"  extracted_case={card.get('id')}|{cname}", flush=True)

    # Preserve original order (by case_id position in the input list)
    id_order = {cid: i for i, cid in enumerate(case_ids)}
    results.sort(key=lambda c: id_order.get(c["id"], 1_000_000))

    return "CASE_BRIEFS_JSON: " + json.dumps(results, indent=2, default=str)


## Test Tool 4 directly

In [ ]:
def parse_related_case_ids(graph_output: str) -> list[int]:
    m = re.search(r"^RELATED_CASE_IDS:\s*(.+)$", graph_output, flags=re.MULTILINE)
    if not m:
        return []
    raw = m.group(1)
    ids = []
    for part in raw.split(","):
        part = part.strip()
        if part.isdigit():
            ids.append(int(part))
    return ids


def parse_ids(tool_output: str) -> list[int]:
    ids = []
    for line in tool_output.splitlines():
        m = re.match(r"^(\d+)\s+\|", line)
        if m:
            ids.append(int(m.group(1)))
    return ids

# Example: run Tool 1 and Tool 2 quickly (or reuse saved strings)
kw_out = keyword_case_search(
    "'common enemy'~3 AND (surface water OR drainage OR stormwater)",
    court_level="Washington Supreme Court",
    min_year=1900,
)

sem_out = semantic_case_search(
    "developer regraded lot causing flooding; city changed road drainage",
    court_level="Both",   # "Both" = search across both courts (required arg)
    min_year=1900,        # 1900 = effectively no cutoff (required arg)
)

seed_ids = sorted(set(parse_ids(kw_out) + parse_ids(sem_out)))
print("Seed ids:", seed_ids)

print("")
print("// Tool 3: Graph traversal //")
graph_out = precedent_graph_search(seed_case_ids=seed_ids)
print(graph_out)

print("// Tool 4: Case analyst extraction (from Tool 3 output) //")

case_ids = parse_related_case_ids(graph_out)
print("Parsed case_ids:", case_ids[:10], "..." if len(case_ids) > 10 else "")

analysis_json = case_analyst_extract(
    case_ids=case_ids    
)

print("")
print("// Tool 4 Output (formatted) //")

if analysis_json.startswith("CASE_BRIEFS_JSON:"):
    json_str = analysis_json.replace("CASE_BRIEFS_JSON: ", "")
    data = json.loads(json_str)

    # Pretty JSON (full structure)
    print("\n=== Full JSON (pretty) ===")
    print(json.dumps(data, indent=2, ensure_ascii=False))
else:
    print("Unexpected output format:")
    print(analysis_json[:1000])


## Re-assemble with Tool 4 and test run Agent

In [ ]:
client = OpenAIChatClient(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_KEY"],
)

agent = client.as_agent(
    instructions=(
        "You are a helpful legal assistant. Respond with the case names, why each is relevant. Run each tool only once."
    ),
    tools=[
        keyword_case_search,
        semantic_case_search,
        precedent_graph_search,
        case_analyst_extract
    ]
)

user_query = (
    "I have a client in Seattle, WA whose property keeps flooding after the developer "
    "next door regraded their lot and the city redid the drainage on the road."
)

print("// Functions the Agent Called: //")
result = await agent.run(user_query)

print("")
print("// Agent Response: //")
Markdown(result.text)

## Tool 5

In [ ]:
@tool(description=(
    "Retrieve historical precipitation data for a location and date range to support "
    "weather-related legal claims (flooding, storm damage, drainage disputes). "
    "Uses the Open-Meteo Archive API. This tool is OPTIONAL — only call it when the "
    "legal question involves weather, water, storms, flooding, drainage, or precipitation."
))
def get_weather_evidence(
    latitude: Annotated[float, Field(
        description="Latitude of the location (WGS84). Example: 47.6062 for Seattle."
    )],
    longitude: Annotated[float, Field(
        description="Longitude of the location (WGS84). Example: -122.3321 for Seattle."
    )],
    start_date: Annotated[str, Field(
        description="Start date in YYYY-MM-DD format."
    )],
    end_date: Annotated[Optional[str], Field(
        default=None,
        description="End date in YYYY-MM-DD format. Defaults to start_date (single day)."
    )] = None,
) -> str:
    print("get_weather_evidence was called")
    print(f"  lat={latitude}  lon={longitude}  start={start_date}  end={end_date or start_date}")

    if end_date is None:
        end_date = start_date

    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "daily": "precipitation_sum",
        "timezone": "UTC",
    }

    try:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        return f"WEATHER_EVIDENCE_ERROR: Failed to retrieve weather data: {e}"

    dates = data.get("daily", {}).get("time", [])
    precip = data.get("daily", {}).get("precipitation_sum", [])

    if not dates:
        return "WEATHER_EVIDENCE: No data returned for the given parameters."

    records = []
    total_mm = 0.0
    for d, p in zip(dates, precip):
        rain = p if p is not None else 0.0
        total_mm += rain
        records.append({"date": d, "precipitation_mm": rain})

    summary = {
        "location": {"latitude": latitude, "longitude": longitude},
        "period": {"start": start_date, "end": end_date},
        "total_precipitation_mm": round(total_mm, 2),
        "days_with_rain": sum(1 for r in records if r["precipitation_mm"] > 0),
        "total_days": len(records),
        "daily_records": records if len(records) <= 14 else records[:7] + records[-7:],
    }

    return "WEATHER_EVIDENCE_JSON: " + json.dumps(summary, ensure_ascii=False)

### Test Tool 5 directly

In [ ]:
print("// Tool 5: Weather Evidence (optional - weather-related query) //")

weather_out = get_weather_evidence(
    latitude=47.6062,
    longitude=-122.3321,
    start_date="2023-01-10",
    end_date="2023-01-20",
)

if weather_out.startswith("WEATHER_EVIDENCE_JSON:"):
    wdata = json.loads(weather_out.replace("WEATHER_EVIDENCE_JSON: ", ""))
    print("\n=== Weather Evidence (pretty) ===")
    print(json.dumps(wdata, indent=2, ensure_ascii=False))
else:
    print(weather_out)


### Re-assemble Agent and test all 5 tools together.

In [ ]:
client = OpenAIChatClient(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_KEY"],
)

agent = client.as_agent(
    instructions=(
        "You are Counsel, an AI legal associate helping analyze Washington State legal cases.\n\n"

        "You have access to a multi-step legal analysis pipeline.\n\n"

        "Required Workflow (always run these 4 tools in order):\n"
        "1. Use keyword_case_search to identify the governing legal doctrine and canonical cases.\n"
        "2. Use semantic_case_search to find factually similar cases using vector similarity.\n"
        "3. Combine results and use precedent_graph_search to identify the strongest line of authority.\n"
        "4. Use case_analyst_extract to extract structured legal facts from the most relevant cases.\n\n"

        "Weather / Storm / Flooding Tool:\n"
        "5. If the question involves weather, flooding, storms, precipitation, drainage, or water damage,\n"
        "   use get_weather_evidence to retrieve historical precipitation data for the relevant location and date.\n"
        "   Include this data as supporting evidence in your analysis.\n\n"

        "Important constraints:\n"
        "- DO NOT skip the required steps (1-4).\n"
        "- DO NOT manually summarize cases without using tools.\n"
        "- Always pass outputs between tools (e.g., case ids, JSON).\n"
        "- Call each tool AT MOST ONCE. Never call the same tool twice.\n\n"

        "Final Output:\n"
        "After all tool calls complete, synthesize the results into a comprehensive legal analysis.\n"
        "Use IRAC format (Issue, Rule, Application, Conclusion).\n"
        "Cite case names from the extracted data. If weather evidence was gathered, incorporate it.\n"
        "Do NOT include raw tool outputs — present a polished analysis.\n\n"

        "Your job is to orchestrate the tools, then produce the final analysis yourself."
    ),
    tools=[
        keyword_case_search,
        semantic_case_search,
        precedent_graph_search,
        case_analyst_extract,
        get_weather_evidence
    ]
)

user_query = (
    "I have a client in Seattle, Washington, whose property floods after a neighboring developer "
    "regraded their lot and the city modified road drainage. Analyze potential legal liability "
    "and produce a structured legal analysis and draft brief."
)

print("// Functions the Agent Called: //")
result = await agent.run(user_query)

print("")
print("// Agent Response: //")
Markdown(result.text)

### Integrate Agent into an app UI with all 5 Tools running together

In [ ]:
# ── Create the Counsel Agent for the UI ──────────────────────────────

_ui_client = OpenAIChatClient(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_KEY"],
)

_ui_agent = _ui_client.as_agent(
    instructions=(
        "You are Counsel, an AI legal research assistant specializing in Washington State case law.\n\n"

        "You have access to a multi-step legal analysis pipeline.\n\n"

        "Required Workflow (always run these 4 tools in order):\n"
        "1. keyword_case_search - BM25 full-text search for canonical cases\n"
        "2. semantic_case_search - DiskANN vector search for factually similar cases\n"
        "3. precedent_graph_search - Citation graph traversal via Apache AGE\n"
        "4. case_analyst_extract - Extract structured legal facts via azure_ai.extract\n\n"

        "Weather / Storm / Flooding Tool:\n"
        "5. Always run this tool last if ran. If the question involves weather, flooding, storms, precipitation, drainage, or water damage,\n"
        "   use get_weather_evidence to retrieve historical precipitation data for the relevant location and date.\n"
        "   Include this data as supporting evidence in your analysis.\n\n"

        "Constraints:\n"
        "- Do NOT skip the required steps (1-4).\n"
        "- If any mention of flooding, storms, drainage, or water damage is detected, you must use the Weather / Storm / Flooding Tool. And run this tool last.\n"
        "- Pass outputs between tools (case ids, JSON payloads).\n"
        "- Do NOT manually summarize without tools.\n"
        "- Call each tool AT MOST ONCE. Never call the same tool twice.\n\n"

        "Final Response Format (Markdown):\n"
        "After all tool calls complete, synthesize the results into a polished legal analysis.\n"
        "- A short opening paragraph summarizing the key legal holding and its significance.\n"
        "- A short paragraph distinguishing the most important prior cases.\n"
        "- If weather evidence was gathered, a paragraph incorporating precipitation data as supporting evidence.\n"
        "- **Key Takeaway:** A concise statement of the principal legal rule.\n"
        "- **Key Cases in this Line of Authority:** A bulleted list of relevant cases, each with case name, citation, and year.\n"
        "- **Statutes Extracted:** A bulleted list of statutes and what case it is from.\n"
    ),
    tools=[
        keyword_case_search,
        semantic_case_search,
        precedent_graph_search,
        case_analyst_extract,
        get_weather_evidence,
    ],
)

# ── Tool Trace Metadata ──────────────────────────────────────────────
_TOOL_META = {
    "keyword_case_search":    {"idx": 1, "name": "KeywordCaseSearchTool",  "tech": "BM25 Search"},
    "semantic_case_search":   {"idx": 2, "name": "SemanticCaseSearchTool", "tech": "DiskANN with Advanced Filtering"},
    "precedent_graph_search": {"idx": 3, "name": "PrecedentGraphTool",     "tech": "AGE Graph"},
    "case_analyst_extract":   {"idx": 4, "name": "CaseAnalystTool",        "tech": "AI Function azure_ai.extract()"},
    "get_weather_evidence":   {"idx": 5, "name": "WeatherEvidenceTool",    "tech": "Weather External API"},
}

_TRACE_PALETTE = {
    "accent": "#5b5ef7",
    "accent_soft": "#eef0ff",
    "accent_border": "#d8ddff",
    "text": "#1f2340",
    "muted": "#687087",
    "panel": "#ffffff",
    "line": "#dfe3f0",
    "success": "#217346",
    "success_soft": "#edf8f0",
    "success_border": "#cce8d5",
    "success_text": "#1f5f3c",
    "shadow": "0 10px 28px rgba(31, 35, 64, 0.08)",
}

_UI_THEME = gr.themes.Soft(
    primary_hue="indigo",
    neutral_hue="gray",
    radius_size=gr.themes.sizes.radius_lg,
)


def _tool_icon_svg(key: str) -> str:
    accent = _TRACE_PALETTE["accent"]
    icons = {
        "keyword_case_search": (
            f'<svg width="18" height="18" viewBox="0 0 20 20" fill="none" '
            f'xmlns="http://www.w3.org/2000/svg" aria-hidden="true">'
            f'<circle cx="8.5" cy="8.5" r="5.5" stroke="{accent}" stroke-width="1.8"/>'
            f'<path d="M12.5 12.5L16.5 16.5" stroke="{accent}" stroke-width="1.8" stroke-linecap="round"/>'
            f'</svg>'
        ),
        "semantic_case_search": (
            f'<svg width="18" height="18" viewBox="0 0 20 20" fill="none" '
            f'xmlns="http://www.w3.org/2000/svg" aria-hidden="true">'
            f'<ellipse cx="10" cy="5" rx="5.8" ry="2.6" stroke="{accent}" stroke-width="1.6"/>'
            f'<path d="M4.2 5V10.2C4.2 11.6 6.8 12.8 10 12.8C13.2 12.8 15.8 11.6 15.8 10.2V5" stroke="{accent}" stroke-width="1.6"/>'
            f'<path d="M4.2 10.2V15C4.2 16.4 6.8 17.6 10 17.6C13.2 17.6 15.8 16.4 15.8 15V10.2" stroke="{accent}" stroke-width="1.6"/>'
            f'</svg>'
        ),
        "precedent_graph_search": (
            f'<svg width="18" height="18" viewBox="0 0 20 20" fill="none" '
            f'xmlns="http://www.w3.org/2000/svg" aria-hidden="true">'
            f'<circle cx="5" cy="10" r="1.7" stroke="{accent}" stroke-width="1.6"/>'
            f'<circle cx="15" cy="5" r="1.7" stroke="{accent}" stroke-width="1.6"/>'
            f'<circle cx="15" cy="15" r="1.7" stroke="{accent}" stroke-width="1.6"/>'
            f'<path d="M6.5 9L13.2 5.8" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'<path d="M6.5 11L13.2 14.2" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'<path d="M15 6.8V13.2" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'</svg>'
        ),
        "case_analyst_extract": (
            f'<svg width="18" height="18" viewBox="0 0 20 20" fill="none" '
            f'xmlns="http://www.w3.org/2000/svg" aria-hidden="true">'
            f'<path d="M6 2.8H11.8L15.5 6.5V16.5C15.5 17.3 14.8 18 14 18H6C5.2 18 4.5 17.3 4.5 16.5V4.3C4.5 3.5 5.2 2.8 6 2.8Z" stroke="{accent}" stroke-width="1.6"/>'
            f'<path d="M11.5 2.8V6.4H15.2" stroke="{accent}" stroke-width="1.6"/>'
            f'<path d="M7.5 10H12.5" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'<path d="M7.5 13H11.2" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'</svg>'
        ),
        "get_weather_evidence": (
            f'<svg width="18" height="18" viewBox="0 0 20 20" fill="none" '
            f'xmlns="http://www.w3.org/2000/svg" aria-hidden="true">'
            f'<path d="M6 14C3.8 14 2 12.4 2 10.4C2 8.7 3.3 7.3 5 7C5.5 4.7 7.5 3 10 3C12.8 3 15 5 15.3 7.5C17 7.8 18 9.2 18 10.8C18 12.6 16.5 14 14.7 14H6Z" stroke="{accent}" stroke-width="1.6" stroke-linejoin="round"/>'
            f'<path d="M8 16V14" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'<path d="M12 17V14" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'<path d="M10 18V15" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'</svg>'
        ),
    }
    return icons.get(key, "")


def _header_icon_svg() -> str:
    accent = _TRACE_PALETTE["accent"]
    return (
        f'<svg width="28" height="28" viewBox="0 0 24 24" fill="none" '
        f'xmlns="http://www.w3.org/2000/svg" aria-hidden="true">'
        f'<path d="M4 7H20" stroke="{accent}" stroke-width="1.8" stroke-linecap="round"/>'
        f'<path d="M12 5V19" stroke="{accent}" stroke-width="1.8" stroke-linecap="round"/>'
        f'<path d="M7 7C7 10 5.5 12 4 13.3C5 14.1 6.2 14.5 7.5 14.5C9.4 14.5 10.9 13.4 12 11.8" stroke="{accent}" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>'
        f'<path d="M17 7C17 10 18.5 12 20 13.3C19 14.1 17.8 14.5 16.5 14.5C14.6 14.5 13.1 13.4 12 11.8" stroke="{accent}" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>'
        f'<path d="M9 19H15" stroke="{accent}" stroke-width="1.8" stroke-linecap="round"/>'
        f'</svg>'
    )


# ── Trace Parsing Helpers ────────────────────────────────────────────

def _parse_traces(captured: str) -> list[dict]:
    """Parse captured stdout to identify which tools the agent called."""
    traces, lines = [], captured.split("\n")
    for i, line in enumerate(lines):
        for key, meta in _TOOL_META.items():
            if f"{key} was called" in line:
                details = []
                for nxt in lines[i + 1:]:
                    if nxt.startswith("  "):
                        details.append(nxt.strip())
                    else:
                        break
                traces.append({"key": key, "details": details, **meta})
                break
    return traces


def _new_trace_state() -> dict:
    return {
        "traces": [],
        "line_buffer": "",
        "changed": False,
    }


def _ingest_trace_chunk(state: dict, chunk: str, now: float | None = None) -> None:
    """Update trace state as stdout arrives so the UI stays in sync with actual tool starts."""
    if not chunk:
        return

    if now is None:
        now = time.perf_counter()

    state["line_buffer"] += chunk
    while "\n" in state["line_buffer"]:
        line, state["line_buffer"] = state["line_buffer"].split("\n", 1)
        stripped = line.rstrip("\r")

        matched_key = None
        for key in _TOOL_META:
            if f"{key} was called" in stripped:
                matched_key = key
                break

        if matched_key is not None:
            trace = {"key": matched_key, "details": [], "started_at": now, **_TOOL_META[matched_key]}
            state["traces"].append(trace)
            state["changed"] = True
            continue

        if stripped.startswith("  ") and state["traces"]:
            state["traces"][-1]["details"].append(stripped.strip())
            state["changed"] = True


def _run_ui_agent(message: str):
    """Run the async agent on a worker thread so UI updates are not blocked by sync tool calls."""
    return asyncio.run(_ui_agent.run(message))


def _trace_desc(t: dict) -> str:
    d = " ".join(t.get("details", []))
    k = t["key"]
    if k == "keyword_case_search":
        m = re.search(r"query=(.+?)\s\s+court_level=", d)
        if m:
            q = m.group(1).strip().strip("'\"").replace("\\'", "'").replace('\\"', '"')
        else:
            q = "case law terms"
        q = q[:55]
        return f'Searched for "{q}" and related terms.'
    if k == "semantic_case_search":
        parts = []
        m = re.search(r"court_level=([^\s]+)", d)
        if m and m.group(1) != "None":
            parts.append(f"court_level IN ({m.group(1)})")
        m = re.search(r"min_year=(\d+)", d)
        if m:
            parts.append(f"decision_date >= {m.group(1)}")
        filt = ", ".join(parts) if parts else "no additional filters"
        return f"Expanded search using vector similarity with filters: {filt}."
    if k == "precedent_graph_search":
        m = re.search(r"seeds=(\d+)", d)
        n = m.group(1) if m else "N"
        return f"{n} unique cases found via KeywordCaseSearchTool and SemanticCaseSearchTool."
    if k == "case_analyst_extract":
        fm = re.search(r"fields=([^\s]+)", d)
        cm = re.search(r"case_ids=(\d+)", d)
        n_in = cm.group(1) if cm else "the"
        if fm:
            field_list = [f.strip() for f in fm.group(1).split(",") if f.strip()]
            chips = "".join(
                f'<span style="display:inline-block;padding:2px 8px;margin:2px 4px 2px 0;'
                f'background:#eef0ff;border:1px solid #d8ddff;border-radius:999px;'
                f'font-size:11px;color:#4f56c7;">{f}</span>'
                for f in field_list
            )
            return (
                f'Extracting structured legal entities from {n_in} case opinions:'
                f'<div style="margin-top:6px;">{chips}</div>'
            )
        return f"Extracting structured fields from {n_in} key opinions."
    if k == "get_weather_evidence":
        m = re.search(r"start=(\S+)", d)
        dt = m.group(1) if m else "requested dates"
        return f"Retrieved historical precipitation data for {dt}."
    return ""


def _trace_result(t: dict) -> str:
    d = " ".join(t.get("details", []))
    k = t["key"]
    if k == "keyword_case_search":
        return "Results: 5 cases"
    if k == "semantic_case_search":
        return "Results: 5 cases"
    if k == "precedent_graph_search":
        rm = re.search(r"related_count=(\d+)", d)
        hm = re.search(r"hops=(\d+)", d)
        if rm:
            n = rm.group(1)
            hops = hm.group(1) if hm else "2"
            return f"{n} additional related cases found traversing the graph using {hops} hops"
        return "Paths found via graph traversal"
    if k == "case_analyst_extract":
        cases = re.findall(r"extracted_case=(\d+)\|(.+?)(?=\s+extracted_case=|\s+\w+=|$)", d)
        if cases:
            items = "".join(
                f'<div style="display:flex;align-items:flex-start;gap:6px;margin-top:4px;">'
                f'<span style="color:#217346;font-weight:700;">✓</span>'
                f'<span style="font-size:12px;color:#1f2340;">{name.strip()} '
                f'<span style="color:#687087;">(id {cid})</span></span>'
                f'</div>'
                for cid, name in cases
            )
            return (
                f'<div style="font-weight:600;color:#1f5f3c;margin-top:4px;">'
                f'Extracted {len(cases)} case{"s" if len(cases) != 1 else ""}:</div>'
                f'{items}'
            )
        m = re.search(r"case_ids=(\d+)", d)
        return f"Cases analyzed: {m.group(1) if m else 'N'}"
    return ""


# ── Trace Panel HTML Builder (Light Theme + Flat Icons) ─────────────

def _build_trace_html(traces, t0, running=False, total_time=None):
    """Build trace panel HTML. Supports live updates during streaming."""
    panel = _TRACE_PALETTE["panel"]
    text = _TRACE_PALETTE["text"]
    muted = _TRACE_PALETTE["muted"]
    accent = _TRACE_PALETTE["accent"]
    accent_soft = _TRACE_PALETTE["accent_soft"]
    accent_border = _TRACE_PALETTE["accent_border"]
    line = _TRACE_PALETTE["line"]
    success = _TRACE_PALETTE["success"]
    success_soft = _TRACE_PALETTE["success_soft"]
    success_border = _TRACE_PALETTE["success_border"]
    success_text = _TRACE_PALETTE["success_text"]
    shadow = _TRACE_PALETTE["shadow"]
    now = time.perf_counter()

    if not traces and not running:
        return (
            f'<div style="padding:24px 22px;text-align:center;color:{muted};background:{panel};">'
            f'<div style="width:52px;height:52px;border-radius:16px;background:{accent_soft};margin:0 auto 12px;'
            f'display:flex;align-items:center;justify-content:center;color:{accent};font-size:22px;">⌕</div>'
            f'<div style="font-weight:600;color:{text};margin-bottom:4px;">Tool Trace</div>'
            f'<div>Ask a question to see the research steps</div>'
            f'</div>'
        )
    if not traces and running:
        return (
            f'<div style="padding:24px 22px;text-align:center;color:{muted};background:{panel};">'
            f'<div style="width:52px;height:52px;border-radius:16px;background:{accent_soft};margin:0 auto 12px;'
            f'display:flex;align-items:center;justify-content:center;color:{accent};font-size:22px;" class="spin">◌</div>'
            f'<div style="font-weight:600;color:{text};margin-bottom:4px;">Starting analysis pipeline</div>'
            f'<div>Preparing the first tool call</div>'
            f'</div>'
        )

    n_label = f'{len(traces)} tool{"s" if len(traces) != 1 else ""}'
    state = "running" if running else "used"
    html = (
        f'<div style="background:{panel};padding:2px 2px 8px 2px;">'
        f'<div style="font-weight:700;font-size:15px;color:{text};margin-bottom:14px;">'
        f'Tool Trace <span style="font-weight:500;color:{muted};">({n_label} {state})</span></div>'
    )

    for i, t in enumerate(traces):
        is_last = i == len(traces) - 1
        is_active = running and is_last

        started_at = t.get("started_at", t0)
        if is_active:
            dur = now - started_at
        elif i + 1 < len(traces):
            dur = traces[i + 1].get("started_at", started_at) - started_at
        else:
            end_at = t0 + (total_time or 0)
            dur = end_at - started_at
        dur = max(dur, 0)

        ts = f"{int(dur * 1000)} ms" if dur < 1 else f"{dur:.1f}s"
        desc = _trace_desc(t)
        res = _trace_result(t)
        icon = _tool_icon_svg(t["key"])

        if is_active:
            status_icon = f'<span style="color:{accent};font-size:14px;" class="spin">◌</span>'
            row_bg = "#f8f9ff"
        else:
            status_icon = f'<span style="color:{success};font-size:14px;">●</span>'
            row_bg = panel

        connector = ""
        if i < len(traces) - 1:
            connector = (
                f'<div style="position:absolute;left:15px;top:34px;width:2px;height:40px;'
                f'background:{accent_border};border-radius:999px;"></div>'
            )

        res_div = (
            f'<div style="color:{muted};font-size:13px;margin-top:6px;">{res}</div>'
            if res and (not is_active or t["key"] == "case_analyst_extract") else ""
        )

        html += (
            f'<div style="display:flex;gap:12px;position:relative;margin-bottom:14px;">'
            f'<div style="position:relative;width:32px;flex:0 0 32px;display:flex;justify-content:center;">'
            f'<div style="width:30px;height:30px;border-radius:999px;background:{accent};color:#fff;' 
            f'display:flex;align-items:center;justify-content:center;font-weight:700;font-size:13px;">{t["idx"]}</div>'
            f'{connector}</div>'
            f'<div style="flex:1;background:{row_bg};border:1px solid {line};border-radius:14px;padding:12px 14px;'
            f'box-shadow:{shadow};">'
            f'<div style="display:flex;justify-content:space-between;align-items:flex-start;gap:12px;">'
            f'<div style="display:flex;gap:10px;align-items:flex-start;min-width:0;">'
            f'<div style="width:20px;height:20px;display:flex;align-items:center;justify-content:center;flex:0 0 20px;">{icon}</div>'
            f'<div style="min-width:0;">'
            f'<div style="font-weight:700;color:{text};line-height:1.25;">{t["name"]} '
            f'<span style="font-weight:500;color:{muted};font-size:12px;">({t["tech"]})</span></div>'
            f'<div style="color:{muted};font-size:13px;line-height:1.45;margin-top:4px;">{desc}</div>'
            f'{res_div}</div></div>'
            f'<div style="text-align:right;white-space:nowrap;color:{muted};font-size:13px;">{ts}<div style="margin-top:4px;">{status_icon}</div></div>'
            f'</div></div></div>'
        )

    if running:
        elapsed = now - t0
        html += (
            f'<div style="display:flex;align-items:center;gap:8px;padding:12px 14px;border-radius:12px;' 
            f'background:{accent_soft};border:1px solid {accent_border};">'
            f'<span class="spin" style="color:{text};">◌</span>'
            f'<span style="font-weight:700;color:{text};">Running... {elapsed:.1f}s elapsed</span></div>'
        )
    else:
        t_disp = total_time or 0
        html += (
            f'<div style="display:flex;align-items:center;gap:8px;padding:12px 14px;border-radius:12px;' 
            f'background:{success_soft};border:1px solid {success_border};color:{success_text};">'
            f'<span style="font-size:14px;color:{success};"></span>'
            f'<span style="font-weight:700;color:{success_text};">Completed in {t_disp:.2f} seconds</span></div>'
        )

    html += "</div>"
    return html


# ── Chat Handler (async generator for live streaming) ────────────────

async def _chat_fn(message: str, history: list):
    if not message or not message.strip():
        yield history, _build_trace_html([], 0), ""
        return

    old_stdout = sys.stdout
    buf = io.StringIO()
    trace_state = _new_trace_state()

    class _Tee:
        def write(self, s):
            buf.write(s)
            _ingest_trace_chunk(trace_state, s)
            old_stdout.write(s)
        def flush(self):
            buf.flush()
            old_stdout.flush()
        def __getattr__(self, name):
            return getattr(old_stdout, name)
    sys.stdout = _Tee()

    t0 = time.perf_counter()
    pending_history = history + [{"role": "user", "content": message}]
    yield pending_history, _build_trace_html([], t0, running=True), ""

    task = asyncio.create_task(asyncio.to_thread(_run_ui_agent, message))
    last_render = 0.0

    while not task.done():
        await asyncio.sleep(0.15)
        now = time.perf_counter()
        should_render = trace_state["changed"] or trace_state["traces"] or (now - last_render >= 0.5)
        if not should_render:
            continue

        trace_state["changed"] = False
        last_render = now
        trace_html = _build_trace_html(trace_state["traces"], t0, running=True)
        yield pending_history, trace_html, ""

    elapsed = time.perf_counter() - t0
    sys.stdout = old_stdout

    try:
        res = task.result()
        response = res.text
    except Exception as e:
        response = f"⚠️ An error occurred: {e}"

    parsed = _parse_traces(buf.getvalue())
    if len(parsed) > len(trace_state["traces"]):
        for i, parsed_trace in enumerate(parsed):
            if i < len(trace_state["traces"]):
                continue
            parsed_trace["started_at"] = t0 + elapsed
            trace_state["traces"].append(parsed_trace)

    trace_html = _build_trace_html(trace_state["traces"], t0, running=False, total_time=elapsed)

    final_history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": response},
    ]
    yield final_history, trace_html, ""


# ── Status Bar ───────────────────────────────────────────────────────

def _status_html() -> str:
    now = datetime.now().strftime("%I:%M %p")
    text = _TRACE_PALETTE["muted"]
    success = _TRACE_PALETTE["success"]
    return (
        f'<div style="display:flex;align-items:center;gap:6px;padding:8px 16px;'
        f'font-size:12px;color:{text};flex-wrap:wrap;">'
        f'Connected to HorizonDB (Reader Endpoint) '
        f'<span style="color:{success};">●</span>'
        f'<span style="margin:0 6px;">|</span> Database: {DB_CONFIG.get("dbname", "")}'
        f'<span style="margin:0 6px;">|</span> User: {DB_CONFIG.get("user", "")}'
        f'<span style="margin:0 6px;">|</span> Time: {now}</div>'
    )


# ── Sample Prompts ───────────────────────────────────────────────────

_PROMPT_1 = (
    "I have a client in Seattle, Washington, whose property floods after a "
    "neighboring developer regraded their lot and the city modified road drainage during April and May of 2024."
)
_PROMPT_2 = (
    "What is the standard for establishing a prescriptive easement for "
    "drainage across a neighbor's property in Washington State?"
)
_PROMPT_3 = (
    "Can a municipality in Washington be held liable for surface water damage "
    "caused by changes to public road drainage infrastructure?"
)

    
# ── Build the Gradio App ────────────────────────────────────────────

_CSS = """
:root {
    --counsel-bg: #f6f7fb;
    --counsel-panel: #ffffff;
    --counsel-line: #e2e7f3;
    --counsel-text: #1f2340;
    --counsel-muted: #687087;
    --counsel-accent: #5b5ef7;
    --counsel-accent-soft: #eef0ff;
    --counsel-font: Aptos, "Segoe UI", Tahoma, Geneva, Verdana, sans-serif;
}

.gradio-container,
.gradio-container * {
    font-family: var(--counsel-font) !important;
}

.gradio-container {
    background: linear-gradient(180deg, #fbfbfe 0%, #f4f6fb 100%);
}

.counsel-shell {
    background: transparent;
}

.counsel-hdr {
    padding: 2px 2px 10px 2px;
}

.counsel-hdr h1 {
    margin: 0;
    font-size: 24px;
    display: flex;
    align-items: center;
    gap: 10px;
    color: var(--counsel-text);
}

.counsel-hdr p {
    margin: 4px 0 0;
    font-size: 14px;
    color: var(--counsel-muted);
}

.counsel-header-icon {
    width: 30px;
    height: 30px;
    display: inline-flex;
    align-items: center;
    justify-content: center;
    flex: 0 0 30px;
}

.counsel-chat,
.counsel-trace {
    border: 1px solid var(--counsel-line);
    border-radius: 18px;
    background: var(--counsel-panel);
    box-shadow: 0 16px 34px rgba(31, 35, 64, 0.06);
}

.counsel-chat {
    overflow: hidden;
}

.counsel-chatbot,
.counsel-chatbot > .wrap,
.counsel-chatbot .wrap,
.counsel-chatbot .bubble-wrap,
.counsel-chatbot .message-wrap,
.counsel-chatbot .message-row,
.counsel-chatbot .panel,
.counsel-chatbot .scroll-hide {
    background: #ffffff !important;
}

.counsel-chatbot {
    border: 0 !important;
}

.counsel-chatbot .message,
.counsel-chatbot .message.user,
.counsel-chatbot .message.bot,
.counsel-chatbot .bubble {
    background: transparent !important;
    color: var(--counsel-text) !important;
    border: 0 !important;
    box-shadow: none !important;
}

.counsel-chatbot .message-content {
    background: #ffffff !important;
    color: var(--counsel-text) !important;
    border: 1px solid var(--counsel-line) !important;
    box-shadow: none !important;
}

.counsel-chatbot .message.user {
    background: #f4f6ff !important;
}

.counsel-chatbot .message.bot {
    background: #ffffff !important;
}

.counsel-chatbot .message * {
    color: var(--counsel-text) !important;
}

.counsel-chatbot .avatar-container,
.counsel-chatbot .avatar-container * {
    background: #f4f6ff !important;
    color: var(--counsel-accent) !important;
}

.counsel-trace {
    padding: 18px;
}

.counsel-examples {
    align-items: center;
    gap: 8px;
}

.counsel-examples .gr-button {
    border: 1px solid #dde2ff;
    background: #f7f8ff;
    color: #4f56c7;
    border-radius: 10px;
}

.counsel-input-row {
    gap: 12px;
    align-items: center;
}

.counsel-input,
.counsel-input > div,
.counsel-input .wrap,
.counsel-input .block {
    background: #ffffff !important;
    border-radius: 14px !important;
}

.counsel-input textarea,
.counsel-input input,
.counsel-input-row textarea,
.counsel-input-row input {
    background: #ffffff !important;
    border: 1px solid var(--counsel-line) !important;
    border-radius: 14px !important;
    color: var(--counsel-text) !important;
    box-shadow: none !important;
}

.counsel-input textarea::placeholder,
.counsel-input input::placeholder {
    color: #8f96ab !important;
}

.counsel-send {
    border-radius: 14px !important;
}

footer { display: none !important; }
@keyframes spin { from { transform: rotate(0deg); } to { transform: rotate(360deg); } }
.spin { display: inline-block; animation: spin 1s linear infinite; }
"""

with gr.Blocks(title="Counsel AI – Legal Research Assistant", css=_CSS, theme=_UI_THEME) as demo:
    with gr.Column(elem_classes=["counsel-shell"]):
        gr.HTML(
            '<div class="counsel-hdr">'
            f'<h1><span class="counsel-header-icon">{_header_icon_svg()}</span> Counsel AI - Legal Research Assistant</h1>'
            '<p>Ask legal questions. Counsel uses HorizonDB to find, analyze, and synthesize case law.</p>'
            '</div>'
        )

        with gr.Row(equal_height=True):
            with gr.Column(scale=3, elem_classes=["counsel-chat"]):
                chatbot = gr.Chatbot(height=500, show_label=False, type="messages", elem_classes=["counsel-chatbot"])
            with gr.Column(scale=2, elem_classes=["counsel-trace"]):
                trace_panel = gr.HTML(value=_build_trace_html([], 0))

        with gr.Row(elem_classes=["counsel-examples"]):
            gr.HTML('<span style="font-size:13px;color:#687087;padding:0 4px;">Example questions</span>')
            ex1 = gr.Button("Property flooding from neighbor regrading & city drainage", size="sm")
            ex2 = gr.Button("Prescriptive easement for drainage in WA", size="sm")
            ex3 = gr.Button("Municipal liability for road drainage damage", size="sm")

        with gr.Row(elem_classes=["counsel-input-row"]):
            msg = gr.Textbox(
                placeholder="Ask a legal question...",
                show_label=False,
                scale=5,
                container=False,
                elem_classes=["counsel-input"],
            )
            send_btn = gr.Button("Send", variant="primary", scale=1, elem_classes=["counsel-send"])

        gr.HTML(_status_html())

    send_btn.click(_chat_fn, [msg, chatbot], [chatbot, trace_panel, msg])
    msg.submit(_chat_fn, [msg, chatbot], [chatbot, trace_panel, msg])


    ex1.click(lambda: _PROMPT_1, outputs=msg)
    ex2.click(lambda: _PROMPT_2, outputs=msg)
    ex3.click(lambda: _PROMPT_3, outputs=msg)

gr.close_all()
demo.launch(server_port=7860, inline=False)

### 🎉 Congratulations you have completed the Lab!

#### 🚀 Next Steps

We have curated additional resources to enhance your ongoing journey in building AI agents and AI-powered applications with Azure Database for PostgreSQL.

- A more detailed blog post about the legal case example of lab in the [GraphRAG Solution for Azure Database for PostgreSQL](https://aka.ms/pg-graphrag) check the code in the [GitHub repository](https://aka.ms/postgres-graphrag-solution).
- Learn more about [Graph data in Azure Database for PostgreSQL](https://aka.ms/age-blog).
- Get familiar with the new [PostgreSQL extension for Visual Studio Code]().